In [2]:
import numpy as np
from nilearn import image
# 註：已經不需要 import datasets，因為我們直接讀取本機檔案

# ==========================================
# 1. 設定檔案路徑
# ==========================================
# 請替換為您要分析的 Z-score 3D 影像路徑
img_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/ZMAP/0mm/roi1_HIP_R/result/mean_ses2-1.nii' 

# 您的 AAL3 圖譜與標籤路徑
aal_img_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii'
aal_txt_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii.txt'

# ==========================================
# 2. 處理統計影像 (尋找最大值)
# ==========================================
img = image.load_img(img_path)
data = img.get_fdata()

# 尋找全腦最大值與對應的 Voxel Index (避開 NaN)
max_val = np.nanmax(data)
max_idx = np.unravel_index(np.nanargmax(data), data.shape)

# 透過 Affine 矩陣將 Voxel Index 轉換為 MNI 空間座標
mni_coords = image.coord_transform(max_idx[0], max_idx[1], max_idx[2], img.affine)

# ==========================================
# 3. 處理 AAL3 圖譜與標籤
# ==========================================
aal_img = image.load_img(aal_img_path)

# 建立標籤字典 (解析 txt 檔)
aal_labels = {}
with open(aal_txt_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        parts = line.strip().split()
        if not parts:
            continue
        
        # 判斷第一欄是不是數字 ID (例如 "1 Precentral_L")
        if parts[0].isdigit():
            roi_id = int(parts[0])
            roi_name = " ".join(parts[1:]) # 支援名稱中帶有空格的情況
        else:
            # 如果沒有數字 ID，假設行號 (從 1 開始) 就是 ID
            roi_id = i + 1
            roi_name = line.strip()
        
        aal_labels[roi_id] = roi_name

# ==========================================
# 4. 座標反查與結果輸出
# ==========================================
# 將 MNI 座標轉換為 AAL3 圖譜的 Voxel 座標 (因解析度可能與輸入影像不同)
inv_affine = np.linalg.inv(aal_img.affine)
aal_vox = image.coord_transform(mni_coords[0], mni_coords[1], mni_coords[2], inv_affine)
aal_vox = np.round(aal_vox).astype(int)

# 提取該座標在 AAL3 圖譜中的數值 (Region ID)
region_id = int(aal_img.get_fdata()[aal_vox[0], aal_vox[1], aal_vox[2]])

# 對照字典取得名稱
if region_id > 0:
    region_name = aal_labels.get(region_id, f"Unknown_Region_ID_{region_id}")
else:
    region_name = "Background / 腦外區域"

print("-" * 50)
print(f"最大數值 (Z-score): {max_val:.4f}")
print(f"MNI 座標: ({mni_coords[0]:.1f}, {mni_coords[1]:.1f}, {mni_coords[2]:.1f})")
print(f"AAL3 腦區: {region_name} (ID: {region_id})")
print("-" * 50)

--------------------------------------------------
最大數值 (Z-score): 0.1025
MNI 座標: (27.0, -33.0, 69.0)
AAL3 腦區: Postcentral_R 62 (ID: 62)
--------------------------------------------------


In [3]:
# ==========================================
# 前置作業：假設 img, data, aal_img, inv_affine, aal_labels 都已讀取完畢
# ==========================================

# 請替換為您要分析的 Z-score 3D 影像路徑
img_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/ZMAP/0mm/roi1_HIP_R/result/roi1_ses01_mean.nii' 

# 您的 AAL3 圖譜與標籤路徑
aal_img_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii'
aal_txt_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii.txt'

# 1. 將 3D 矩陣攤平為 1D 陣列
flat_data = data.flatten()

# 2. 找出所有非 NaN 數值的索引 (避免 NaN 破壞排序)
valid_indices = np.where(~np.isnan(flat_data))[0]
valid_values = flat_data[valid_indices]

# 3. 對有效數值進行從小到大的排序，取得排序後的「索引」
sorted_order = np.argsort(valid_values)
sorted_valid_indices = valid_indices[sorted_order]

# 4. 取出最小的 5 個與最大的 5 個 (最大值將其反轉，讓第一名排最前面)
bottom_5_indices = sorted_valid_indices[:15]
top_5_indices = sorted_valid_indices[-15:][::-1]

# ==========================================
# 建立輔助函數：處理座標轉換並輸出漂亮表格
# ==========================================
def print_extremes(indices, title):
    print(f"\n【 {title} 】")
    print("-" * 75)
    print(f"{'Z-score':>8} | {'MNI (X, Y, Z)':^22} | {'AAL3 腦區 (ID)'}")
    print("-" * 75)
    
    for flat_idx in indices:
        # 取出該點的 Z-score 數值
        val = flat_data[flat_idx]
        
        # 將 1D 索引轉回原本的 3D 體素矩陣座標 (i, j, k)
        idx_3d = np.unravel_index(flat_idx, data.shape)
        
        # 體素座標 -> MNI 座標
        mni = image.coord_transform(idx_3d[0], idx_3d[1], idx_3d[2], img.affine)
        
        # MNI 座標 -> AAL3 圖譜體素座標
        aal_vox = image.coord_transform(mni[0], mni[1], mni[2], inv_affine)
        aal_vox = np.round(aal_vox).astype(int)
        
        # 從 AAL3 圖譜提取 Region ID 並查對字典
        try:
            region_id = int(aal_img.get_fdata()[aal_vox[0], aal_vox[1], aal_vox[2]])
            if region_id > 0:
                region_name = aal_labels.get(region_id, f"Unknown_ID_{region_id}")
            else:
                region_name = "Background / 腦外區域"
        except IndexError:
            # 避免 MNI 座標剛好落在 AAL 圖譜邊界之外導致報錯
            region_id = 0
            region_name = "Out of Bounds (超出圖譜範圍)"

        # 格式化輸出，確保小數點與欄位對齊
        print(f"{val:>8.4f} | ({mni[0]:>5.1f}, {mni[1]:>5.1f}, {mni[2]:>5.1f}) | {region_name} ({region_id})")

# ==========================================
# 執行輸出
# ==========================================
print_extremes(top_5_indices, "Top 5 最大相關性 (最強正相關)")
print_extremes(bottom_5_indices, "Top 5 最小相關性 (最強負相關)")
print("-" * 75)


【 Top 5 最大相關性 (最強正相關) 】
---------------------------------------------------------------------------
 Z-score |     MNI (X, Y, Z)      | AAL3 腦區 (ID)
---------------------------------------------------------------------------
  0.1025 | ( 27.0, -33.0,  69.0) | Postcentral_R 62 (62)
  0.0990 | (-21.0,   9.0,  66.0) | Frontal_Sup_2_L 3 (3)
  0.0931 | ( 27.0,  54.0, -12.0) | Frontal_Mid_2_R 6 (6)
  0.0907 | ( 42.0,  54.0,  -9.0) | Frontal_Mid_2_R 6 (6)
  0.0896 | ( 18.0,  57.0,  30.0) | Frontal_Sup_2_R 4 (4)
  0.0870 | (-42.0, -30.0,  60.0) | Postcentral_L 61 (61)
  0.0868 | ( 27.0, -75.0, -51.0) | Cerebellum_8_R 108 (108)
  0.0867 | (-36.0, -69.0, -24.0) | Cerebellum_Crus1_L 95 (95)
  0.0865 | ( 27.0, -30.0,  69.0) | Precentral_R 2 (2)
  0.0853 | (-15.0, -72.0, -48.0) | Cerebellum_8_L 107 (107)
  0.0844 | ( 57.0, -12.0,  45.0) | Precentral_R 2 (2)
  0.0838 | ( 12.0, -39.0, -21.0) | Cerebellum_3_R 100 (100)
  0.0832 | ( -6.0, -48.0, -18.0) | Cerebellum_4_5_L 101 (101)
  0.0824 | ( -9.0,  

In [5]:
from nilearn import plotting
import numpy as np
from nilearn.plotting import view_img

# ==========================================
# 1. 建立清單來儲存這 5 個點的座標與標籤
# ==========================================
top_5_mni = []
top_5_labels = []

# (假設您已經執行了前面的極值尋找，取得了 top_5_indices)
for flat_idx in top_5_indices:
    idx_3d = np.unravel_index(flat_idx, data.shape)
    mni = image.coord_transform(idx_3d[0], idx_3d[1], idx_3d[2], img.affine)
    
    # 座標反查 (同之前的邏輯)
    aal_vox = image.coord_transform(mni[0], mni[1], mni[2], inv_affine)
    aal_vox = np.round(aal_vox).astype(int)
    try:
        region_id = int(aal_img.get_fdata()[aal_vox[0], aal_vox[1], aal_vox[2]])
        region_name = aal_labels.get(region_id, "Unknown") if region_id > 0 else "Background"
    except IndexError:
        region_name = "Out of Bounds"
        
    # 將座標與名稱存入清單
    top_5_mni.append([mni[0], mni[1], mni[2]])
    top_5_labels.append(region_name)

# ==========================================
# 方法 A：視覺化「這五個點」(彩色球體)
# ==========================================
html_markers = plotting.view_markers(
    top_5_mni,
    marker_labels=top_5_labels,
    marker_color='red',  # 也可以給一個顏色清單 ['red', 'blue', 'green', ...]
    marker_size=10,
    title="Top 5 Z-score Peaks"
)

# 儲存為互動式網頁
html_markers.save_as_html("top_5_points.html")
print("✅ 已生成 top_5_points.html (3D 標點圖)")

# ==========================================
# 方法 B：使用 view_img 視覺化「立體統計地圖」
# ==========================================
# 將十字游標 (cut_coords) 設定在第一名的座標上
html_img = plotting.view_img(
    img, 
    threshold=3.1,             # 隱藏 Z-score 較低的雜訊區域 (可依需求調整)
    cut_coords=top_5_mni[0],   # 對準第一個點 (絕對最大值)
    title="Z-score Map (Crosshair on Max Peak)",
    cmap='cold_hot'            # 適合呈現正負相關性的冷暖色階
)

html_img.save_as_html("top_5_view_img.html")
print("✅ 已生成 top_5_view_img.html (3D 切面圖)")
view_img(html_img)

✅ 已生成 top_5_points.html (3D 標點圖)


/tmp/ipykernel_4027808/895312413.py:48: UserWarning: The given float value must not exceed 0.11612305045127869. But, you have given threshold=3.1.
  html_img = plotting.view_img(
/tmp/ipykernel_4027808/895312413.py:48: UserWarning: The given float value must not exceed 0.0. But, you have given threshold=3.1.
  html_img = plotting.view_img(
/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/.venv/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


✅ 已生成 top_5_view_img.html (3D 切面圖)


TypeError: input should be a NiftiLike object or an iterable of NiftiLike object. Got: StatMapView

In [6]:
import os


# 2. 設定 labels
# 請將這裡的 .txt 檔名替換為您解壓縮出來的實際標籤檔名
label_path = "/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii.txt"
maps=label_path
labels = ["Background"] # 預設 index 0 為背景

# 讀取標籤檔
try:
    with open(label_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                # AAL 的 txt 檔通常格式為："縮寫 數字代碼 索引" 或類似格式
                # 這裡假設腦區名稱在第一或第二個欄位，請根據您的 txt 檔實際內容調整 index (例如 parts[0] 或 parts[1])
                # 以 AAL3 為例，通常腦區名稱字串是不帶空白的
                
                # 這裡抓取長度大於 2 的字串作為標籤名，避開純數字
                region_name = next((p for p in parts if not p.isdigit()), "Unknown")
                labels.append(region_name)
                
    print("成功載入標籤，前 5 個為:")
    print(labels[:5])
    print(f"總共載入 {len(labels)} 個標籤 (包含背景)")

except FileNotFoundError:
    print(f"找不到標籤檔：{label_path}，請檢查路徑與檔名是否正確！")

# 3. (選擇性) 將它們打包成像 nilearn 回傳的物件格式
# 這樣您後續的程式碼就不需要大改
class CustomAtlas:
    pass

atlas_dataset = CustomAtlas()
atlas_dataset.maps = maps
atlas_dataset.labels = labels


print(atlas_dataset.maps)
print(atlas_dataset.labels)

成功載入標籤，前 5 個為:
['Background', 'Precentral_L', 'Precentral_R', 'Frontal_Sup_2_L', 'Frontal_Sup_2_R']
總共載入 171 個標籤 (包含背景)
/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii.txt
['Background', 'Precentral_L', 'Precentral_R', 'Frontal_Sup_2_L', 'Frontal_Sup_2_R', 'Frontal_Mid_2_L', 'Frontal_Mid_2_R', 'Frontal_Inf_Oper_L', 'Frontal_Inf_Oper_R', 'Frontal_Inf_Tri_L', 'Frontal_Inf_Tri_R', 'Frontal_Inf_Orb_2_L', 'Frontal_Inf_Orb_2_R', 'Rolandic_Oper_L', 'Rolandic_Oper_R', 'Supp_Motor_Area_L', 'Supp_Motor_Area_R', 'Olfactory_L', 'Olfactory_R', 'Frontal_Sup_Medial_L', 'Frontal_Sup_Medial_R', 'Frontal_Med_Orb_L', 'Frontal_Med_Orb_R', 'Rectus_L', 'Rectus_R', 'OFCmed_L', 'OFCmed_R', 'OFCant_L', 'OFCant_R', 'OFCpost_L', 'OFCpost_R', 'OFClat_L', 'OFClat_R', 'Insula_L', 'Insula_R', 'Cingulate_Ant_L', 'Cingulate_Ant_R', 'Cingulate_Mid_L', 'Cingulate_Mid_R', 'Cingulate_Post_L', 'Cingulate_Post_R', 'Hippocampus_L', 'Hippocampus_

In [7]:
import numpy as np
from nilearn import image, datasets
import nibabel as nib

# 1. 讀取您的 3D 影像
img_path = '/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/ZMAP/0mm/roi1_HIP_R/result/roi1_0mm_ses01_mean.nii'
img = image.load_img(img_path)
data = img.get_fdata()

# 2. 尋找全腦最大值與對應的 Voxel Index
# 使用 nanmax/nanargmax 以避開背景的 NaN 值
max_val = np.nanmax(data)
max_idx = np.unravel_index(np.nanargmax(data), data.shape)

# 3. 透過 Affine 矩陣將 Voxel Index 轉換為 MNI 空間座標
mni_coords = image.coord_transform(max_idx[0], max_idx[1], max_idx[2], img.affine)

# 4. 載入 AAL 圖譜 (nilearn 會自動下載並快取)
# aal = datasets.fetch_atlas_aal()
aal_img = image.load_img(atlas_dataset.maps)

# 5. 反查該 MNI 座標在 AAL 圖譜中的腦區
# 將 MNI 座標轉換為 AAL 圖譜的 Voxel 座標 (因為兩張圖的解析度可能不同)
inv_affine = np.linalg.inv(aal_img.affine)
aal_vox = image.coord_transform(mni_coords[0], mni_coords[1], mni_coords[2], inv_affine)
aal_vox = np.round(aal_vox).astype(int)

# 提取該座標的圖譜數值 (Region ID)
region_id = int(aal_img.get_fdata()[aal_vox[0], aal_vox[1], aal_vox[2]])

# 對照 AAL 標籤名稱 (AAL ID 從 1 開始，對應 Python list 的 0)
if region_id > 0:
    region_name = aal.labels[region_id - 1]
else:
    region_name = "Background / 腦室 / 腦外區域"

print("-" * 40)
print(f"最大數值: {max_val:.4f}")
print(f"MNI 座標: ({mni_coords[0]:.1f}, {mni_coords[1]:.1f}, {mni_coords[2]:.1f})")
print(f"AAL 腦區: {region_name}")
print("-" * 40)

ImageFileError: Cannot work out file type of "/bml/projects/07_inference-clinical-trial/projects/07-09_ntsec-lego-fmri-connectivity/data/derivatives/AAL_ATLAS/AAL3/AAL3v1.nii.txt"